In [15]:
source("./scale.R")
test <- read.csv("/data/coro_data.csv")
test = test %>% rename(x = x_section, y = y_section)
library("xtable")
library("tibble")
options(warn=-1)

In [5]:
library('foreach')
library('doParallel')
cores=detectCores()
cl <- makeCluster(cores[1]-1) #not to overload your computer
registerDoParallel(cl)

Loading required package: iterators

Loading required package: parallel



### Feature calculation

In [6]:
ecdf_fun <- function(x,perc) ecdf(x)(perc)
## Calculate column stats for each type
calc_pp_2d<-function(typ,pp,section, rm){
    nnp <- c()
    nni <- c()
    mnn <- c()
    secs <- c()
    res<- list()
    ar <- 0
    cn <- 0
    for (i in 1:length(pp)){     
        perc<- ecdf_fun(nndist(pp[[i]]), 0.01)
        int<- intensity(pp[[i]])
        mval<- mean(marks(pp[[i]]))
        nnp<-append(nnp,perc)
        nni<-append(nni,int)
        mnn <- append(mnn, mval)
        secs<-append(secs, as.character(section[i]))
        ar <- ar + area(pp[[i]])
        cn <- cn + npoints(pp[[i]])
    }   
    res[['section']] = paste(secs, collapse = ', ')
    res[['area']] = round(ar,2)
    res[['perc']] = round(mean(nnp)*100,2)
    res[['lam']] = as.integer(round(mean(nni)))
    res[['conf']] = round(mean(mnn),2)
    res[['range']] = as.integer(round(1000*rm))
    res[['Cluster']] = typ
    res[['n']] = cn
    return(res)
}

### Excitatory clusters
We get the list of types and sections from the filtered table.

In [7]:
df<- read.csv("filter-exc.csv")
df1 <- replace(df, is.na(df), 0)
df1[df1<=0.5] <- NA
df1<-df1[rowSums(is.na(df1)) != ncol(df1)-1, ]
row.names(df1) <- NULL

In [8]:
types = df1$cluster
types

[1] "0042 L6 IT CTX Glut_2"   "0061 L5 IT CTX Glut_3"  
 [3] "0070 L4/5 IT CTX Glut_1" "0072 L4/5 IT CTX Glut_1"
 [5] "0077 L4/5 IT CTX Glut_2" "0078 L4/5 IT CTX Glut_2"
 [7] "0082 L4/5 IT CTX Glut_2" "0084 L4/5 IT CTX Glut_3"
 [9] "0087 L4/5 IT CTX Glut_3" "0088 L4/5 IT CTX Glut_3"
[11] "0091 L4/5 IT CTX Glut_4" "0094 L4/5 IT CTX Glut_4"
[13] "0095 L4/5 IT CTX Glut_5" "0098 L4/5 IT CTX Glut_5"
[15] "0104 L2/3 IT CTX Glut_1" "0105 L2/3 IT CTX Glut_1"
[17] "0109 L2/3 IT CTX Glut_2" "0117 L2/3 IT CTX Glut_4"
[19] "0118 L2/3 IT CTX Glut_4" "0350 L5 ET CTX Glut_1"  
[21] "0351 L5 ET CTX Glut_1"   "0364 L5 ET CTX Glut_2"  
[23] "0436 L6 CT CTX Glut_1"   "0444 L6 CT CTX Glut_2"  
[25] "0468 L5 NP CTX Glut_3"   "0473 L5 NP CTX Glut_4"

In [9]:
sections<- apply(df1[,-1], 1, function(i) colnames(df1[,-1])[ !is.na(i) ]) ## qualified colns
sections<- sapply(sections, function(x){as.numeric(gsub("X.", "", x))})

In [11]:
df = data.frame(matrix(vector(), 0, 8,
                dimnames=list(c(), c('Cluster','section',"area",'range','perc', "lam",'conf','n'))),
                stringsAsFactors=F)
df$Cluster <- as.character(df$Cluster)
df$section <- as.character(df$section)
df

Cluster,section,area,range,perc,lam,conf,n
<chr>,<chr>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>,<lgl>


### Post region-selection  
We use the final window for each cluster and calculate the interaction range.

In [17]:
res <- foreach(t = 0:length(types), .combine=function(x,y) bind_rows(as.data.frame(x),as.data.frame(y)),.packages=c('dplyr','spatstat','poolr')) %dopar% {
    if (t == 0){
        q <- df
    }
    else{
        pp<-create_pp(test, NULL,sections[[t]], types[t])
        rm<-min(as.numeric(quantile(do.call(c,lapply(pp, nndist)),0.5)),0.06) 
        ## write into data frame
        q <- calc_pp_2d(types[t],pp,sections[[t]], rm)
        q
    }
}
res

Cluster,section,area,range,perc,lam,conf,n
<chr>,<chr>,<dbl>,<int>,<dbl>,<int>,<dbl>,<dbl>
0042 L6 IT CTX Glut_2,"59, 60",0.76,29,0.98,318,0.70,260
0061 L5 IT CTX Glut_3,59,0.38,33,3.96,268,0.68,101
0070 L4/5 IT CTX Glut_1,59,0.30,44,2.20,302,0.70,91
0072 L4/5 IT CTX Glut_1,"59, 60, 61",1.25,30,0.66,447,0.72,558
0077 L4/5 IT CTX Glut_2,"60, 61",0.45,32,1.39,427,0.70,194
0078 L4/5 IT CTX Glut_2,"59, 60, 61",0.76,34,1.42,279,0.73,210
0082 L4/5 IT CTX Glut_2,"59, 60, 61",0.54,38,0.00,423,0.71,236
0084 L4/5 IT CTX Glut_3,"59, 60, 61",1.28,38,0.80,335,0.71,430
0087 L4/5 IT CTX Glut_3,"60, 61",0.74,34,1.32,317,0.72,243


In [22]:
print.xtable(xtable(res), file = "./feature-ex.txt")

### Combine with aggregated p-values

In [25]:
pv <- read.csv("./table-ex.csv")

In [22]:
pv<-add_column(pv, pv.adj= signif(p.adjust(pv$pval, method="BH"),2))
pv1<- pv[order(pv$pv.adj), ]
row.names(pv1) <- NULL
pv1

Cluster,X59,X60,X61,pval,perc,lam,conf,rm,pv.adj
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<int>,<dbl>
0072 L4/5 IT CTX Glut_1,0.005,0.005,0.005,1.8e-05,0.66,447,0.72,21,9.0e-05
0084 L4/5 IT CTX Glut_3,0.005,0.005,0.005,1.8e-05,0.80,335,0.71,29,9.0e-05
0091 L4/5 IT CTX Glut_4,0.005,0.005,0.005,1.8e-05,1.31,534,0.71,22,9.0e-05
0095 L4/5 IT CTX Glut_5,0.005,0.005,0.005,1.8e-05,0.78,426,0.72,20,9.0e-05
0105 L2/3 IT CTX Glut_1,0.005,0.005,0.005,1.8e-05,1.29,1216,0.74,10,9.0e-05
0104 L2/3 IT CTX Glut_1,0.005,0.006,0.006,2.5e-05,3.83,875,0.72,6,9.6e-05
0117 L2/3 IT CTX Glut_4,0.008,0.005,0.005,2.7e-05,2.07,560,0.71,22,9.6e-05
0082 L4/5 IT CTX Glut_2,0.005,0.012,0.005,3.9e-05,0.00,423,0.71,29,1.1e-04
0351 L5 ET CTX Glut_1,0.005,0.009,0.007,4.0e-05,0.34,282,0.72,26,1.1e-04


In [14]:
library("xtable")
print.xtable(xtable(pv1,digits=3), file = "./ex-section.txt")

In [30]:
resp<-merge(pv[,c(1,5)],res,by="Cluster")
df<- resp[order(resp$pval), ]
row.names(df) <- NULL
df<-add_column(df, pv.adj= signif(p.adjust(df$pval, method="BH"),2), .after = 3)

In [31]:
drops <- c("pval")
df<-df[ , !(names(df) %in% drops)]
df

Cluster,section,pv.adj,area,range,perc,lam,conf,n
<chr>,<chr>,<dbl>,<dbl>,<int>,<dbl>,<int>,<dbl>,<dbl>
0072 L4/5 IT CTX Glut_1,"59, 60, 61",9.0e-05,1.25,30,0.66,447,0.72,558
0084 L4/5 IT CTX Glut_3,"59, 60, 61",9.0e-05,1.28,38,0.80,335,0.71,430
0091 L4/5 IT CTX Glut_4,"59, 60, 61",9.0e-05,0.97,31,1.31,534,0.71,521
0095 L4/5 IT CTX Glut_5,"59, 60, 61",9.0e-05,1.69,29,0.78,426,0.72,751
0105 L2/3 IT CTX Glut_1,"59, 60, 61",9.0e-05,0.25,19,1.29,1216,0.74,298
0104 L2/3 IT CTX Glut_1,"59, 60, 61",9.6e-05,0.35,16,3.83,875,0.72,324
0117 L2/3 IT CTX Glut_4,"59, 60, 61",9.6e-05,0.62,32,2.07,560,0.71,323
0082 L4/5 IT CTX Glut_2,"59, 60, 61",1.1e-04,0.54,38,0.00,423,0.71,236
0351 L5 ET CTX Glut_1,"59, 60, 61",1.1e-04,1.53,36,0.34,282,0.72,431


In [32]:
print.xtable(xtable(df), file = "./feature-ex.txt")